<a href="https://colab.research.google.com/github/23Kou/PUC_PROJETO_BDII/blob/main/full_text_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 28.8 MB/s eta 0:00:00


In [2]:
import psycopg2
from google.colab import userdata

# Busca a string de conexão salva nos Secrets
DATABASE_URL = userdata.get('DATABASE_URL')

try:
    # Abre a conexão com o banco de dados
    conn = psycopg2.connect(DATABASE_URL)
    cursor = conn.cursor()

    cursor.execute("SELECT version();")
    db_version = cursor.fetchone()

    print("conectado")
    print(f"Versão do Postgres: {db_version[0]}")

except Exception as e:
    print(f"Falha ao conectar: {e}")

conectado
Versão do Postgres: PostgreSQL 18.4 (6e15e70) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [3]:
try:
    # Executa o comando SQL para criar a tabela
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS documentos_texto (
            id          SERIAL PRIMARY KEY,
            uuid        UUID DEFAULT gen_random_uuid() UNIQUE NOT NULL,
            titulo      TEXT NOT NULL,
            conteudo    TEXT NOT NULL,
            ts_conteudo TSVECTOR,
            created_at  TIMESTAMPTZ DEFAULT NOW() NOT NULL,
            updated_at  TIMESTAMPTZ DEFAULT NOW() NOT NULL
        );
    """)

    # Confirma as alterações no banco de dados
    conn.commit()
    print("Tabela criada")

except Exception as e:
    # Em caso de erro, reverte a transação
    conn.rollback()
    print(f"Falha ao criar tabela: {e}")

Tabela criada


### Limpando os dados existentes

Para garantir que apenas o conteúdo em português seja processado, vamos primeiro **limpar a tabela `documentos_texto`** para remover qualquer conteúdo em inglês que já tenha sido inserido. Isso reiniciará o banco de dados para uma nova ingestão.

In [4]:
query_truncate = "TRUNCATE TABLE documentos_texto RESTART IDENTITY CASCADE;"

try:
    cursor.execute(query_truncate)
    conn.commit()
    print("Tabela 'documentos_texto' truncada e reiniciada")
except Exception as e:
    conn.rollback()
    print(f"Falha ao truncar tabela: {e}")

Tabela 'documentos_texto' truncada e reiniciada


In [5]:
import urllib.request

# Links oficiais de texto puro no Project Gutenberg
links_livros = {
    "Dom Casmurro": "https://www.gutenberg.org/files/55752/55752-0.txt",
    "Memórias Póstumas de Brás Cubas": "https://www.gutenberg.org/files/54829/54829-0.txt",
    "O Alienista": "https://www.gutenberg.org/files/55546/55546-0.txt"
}

query_insercao = "INSERT INTO documentos_texto (titulo, conteudo) VALUES (%s, %s);"

for titulo, url in links_livros.items():
    print(f"Baixando '{titulo}'...")
    try:
        with urllib.request.urlopen(url) as response:
            texto = response.read().decode('utf-8')

        # Divide em parágrafos limpos
        linhas = texto.split('\n')
        paragrafos = []
        paragrafo_atual = []

        for linha in linhas:
            linha_limpa = linha.strip()
            if linha_limpa:
                paragrafo_atual.append(linha_limpa)
            else:
                if paragrafo_atual:
                    txt_paragrafo = " ".join(paragrafo_atual)
                    if len(txt_paragrafo) > 50: # Evita linhas curtas ou vazias
                        paragrafos.append(txt_paragrafo)
                    paragrafo_atual = []

        print(f"Enviando {len(paragrafos)} parágrafos de '{titulo}' para o Neon...")
        for i, paragrafo in enumerate(paragrafos):
            titulo_trecho = f"{titulo} - Parágrafo {i+1}"
            cursor.execute(query_insercao, (titulo_trecho, paragrafo))

        conn.commit()
        print(f"'{titulo}' gravado com sucesso!\n")
    except Exception as e:
        conn.rollback()
        print(f"Falha no livro '{titulo}': {e}\n")

print("Todos os livros estão salvos dentro do banco de dados")

Baixando 'Dom Casmurro'...
Enviando 939 parágrafos de 'Dom Casmurro' para o Neon...
'Dom Casmurro' gravado com sucesso!

Baixando 'Memórias Póstumas de Brás Cubas'...
Enviando 786 parágrafos de 'Memórias Póstumas de Brás Cubas' para o Neon...
'Memórias Póstumas de Brás Cubas' gravado com sucesso!

Baixando 'O Alienista'...
Enviando 1126 parágrafos de 'O Alienista' para o Neon...
'O Alienista' gravado com sucesso!

Todos os livros estão salvos dentro do banco de dados


In [14]:
config_busca_textual = """
UPDATE documentos_texto SET ts_conteudo = to_tsvector('portuguese', conteudo);

DROP INDEX IF EXISTS idx_ts_conteudo;
CREATE INDEX idx_ts_conteudo ON documentos_texto USING GIN (ts_conteudo);
"""

try:
    cursor.execute(config_busca_textual)
    conn.commit()
    print("Configuração realizada: ts_conteudo preenchido e índice GIN criado!")
except Exception as e:
    conn.rollback()
    print(f"Falha na configuração de indexação: {e}")

Configuração realizada: ts_conteudo preenchido e índice GIN criado!


In [27]:

q1_teste = """
SELECT titulo, conteudo FROM documentos_texto
WHERE ts_conteudo @@ to_tsquery('portuguese', 'longu:*');
"""
rodar_consulta("1. Teste de Busca por Prefixo (longu:*)", q1_teste)


--- 1. Teste de Busca por Prefixo (longu:*) ---
('Dom Casmurro - Parágrafo 480', "Enxuguei os olhos, posto que de todas as palavras de José Dias uma só me ficasse no coração; foi aquelle _gravissimo._ Vi depois que elle só queria dizer _grave_, mas o uso do superlativo faz a bocca longa, e, por amor do periodo, José Dias fez crescer a minha tristeza. Se achares neste livro algum caso da mesma familia, avisa-me, leitor, para que o emende na segunda edição; nada ha mais feio que dar pernas longuissimas a ideias brevissimas. Enxuguei os olhos, repito, e fui andando, ancioso agora por chegar a casa, e pedir perdão a minha mãe do ruim pensamento que tive. Emfim, chegámos, entramos, subi tremulo os seis degraus da escada, e d'ahi a pouco, debruçado sobre a cama, ouvia as palavras ternas de minha mãe que me apertava muito as mãos, chamando-me seu filho. Estava queimando, os olhos ardiam nos meus, toda ella parecia consumida por um volcão interno. Ajoelhei-me ao pé do leito, mas como este era

In [20]:
# 2. BUSCA COMPOSTA COM OPERADOR AND (&)
q2 = """
SELECT titulo, conteudo FROM documentos_texto
WHERE ts_conteudo @@ to_tsquery('portuguese', 'menina & fruta');
"""
rodar_consulta("2. Operador AND (menina E fruta)", q2)


--- 2. Operador AND (menina E fruta) ---
('Dom Casmurro - Parágrafo 936', 'Agora, porque é que nenhuma dessas caprichosas me fez esquecer a primeira amada do meu coração? Talvez porque nenhuma tinha os olhos de ressaca, nem os de cigana obliqua e dissimulada. Mas não é este propriamente o resto do livro. O resto é saber se a Capitú da praia da Gloria já estava dentro da de Matacavallos, ou se esta foi mudada naquella por effeito de algum caso incidente. Jesus, filho de Sirach, se soubesse dos meus primeiros ciumes, dir-me-hia, como no seu cap. IX, vers. 1: «Não tenhas ciumes de tua mulher para que ella não se metta a enganar-te com a malicia que apprender de ti.» Mas eu creio que não, e tu concordarás commigo; se te lembras bem da Capitú menina, has de reconhecer que uma estava dentro da outra, como a fruta dentro da casca.')


In [21]:
q3 = """
SELECT titulo, conteudo FROM documentos_texto
WHERE ts_conteudo @@ to_tsquery('portuguese', 'gênio | louco');
"""
rodar_consulta("3. Operador OR (gênio OU louco)", q3)


--- 3. Operador OR (gênio OU louco) ---
('Dom Casmurro - Parágrafo 5', "Vivo só, com um creado. A casa em que moro é propria; fil-a construir de proposito, levado de um desejo tão particular que me vexa imprimil-o, mas vá lá. Um dia, ha bastantes annos, lembrou-me reproduzir no Engenho Novo a casa em que me criei na antiga rua de Matacavallos, dando-lhe o mesmo aspecto e economia daquella outra, que desappareceu. Constructor e pintor entenderam bem as indicações que lhes fiz: é o mesmo predio assobradado, tres janellas de frente, varanda ao fundo, as mesmas alcovas e salas. Na principal destas, a pintura do tecto e das paredes é mais ou menos egual, umas grinaldas de flores miudas e grandes passaros que as tomam nos bicos, de espaço a espaço. Nos quatro cantos do tecto as figuras das estações, e ao centro das paredes os medalhões de Cesar, Augusto, Nero e Massinissa, com os nomes por baixo... Não alcanço a razão de taes personagens. Quando fomos para a casa de Matacavallos, já ella es

In [22]:
# 4. OPERADOR NOT (!) - São mas NÃO doido
q4 = """
SELECT titulo, conteudo FROM documentos_texto
WHERE ts_conteudo @@ to_tsquery('portuguese', 'olhos & !ressaca');
"""
rodar_consulta("4. Operador NOT (olhos mas NÃO ressaca)", q4)


--- 4. Operador NOT (olhos mas NÃO ressaca) ---
('Dom Casmurro - Parágrafo 1', 'Uma noite destas, vindo da cidade para o Engenho Novo, encontrei no trem da Central um rapaz aqui do bairro, que eu conheço de vista e de chapéo. Comprimentou-me, sentou-se ao pé de mim, falou da lua e dos ministros, e acabou recitando-me versos. A viagem era curta, e os versos póde ser que não fossem inteiramente maus. Succedeu, porém, que como eu estava cançado, fechei os olhos tres ou quatro vezes; tanto bastou para que elle interrompesse a leitura e mettesse os versos no bolso.')
('Dom Casmurro - Parágrafo 27', 'Nem sempre ia naquelle passo vagaroso e rigido. Tambem se descompunha em accionados, era muita vez rapido e lepido nos movimentos, tão natural nesta como naquella maneira. Outrosim, ria largo, se era preciso, de um grande riso sem vontade, mas communicativo, a tal ponto as bochechas, os dentes, os olhos, toda a cara, todo a pessoa, todo o mundo pareciam rir nelle. Nos lances graves, gravissimo.

In [28]:
# 5. PREFIXO (:*) - Palavras que começam com 'falt'
q5 = """
SELECT titulo, conteudo FROM documentos_texto
WHERE ts_conteudo @@ to_tsquery('portuguese', 'falt:*');
"""
rodar_consulta("5. Prefixo (palavras que começam com 'falt')", q5)


--- 5. Prefixo (palavras que começam com 'falt') ---
('Dom Casmurro - Parágrafo 938', 'Capitulo I         Do titulo II        Do livro III       A denuncia IV        Um dever amarissimo! V         O aggregado VI        Tio Cosme VII       D. Gloria VIII      É tempo! IX        A opera X         Acceito a theoria XI        A promessa XII       Na varanda XIII      Capitú XIV       A inscripção XV        Outra voz repentina XVI       O administrador interino XVII      Os vermes XVIII     Um plano XIX       Sem falta XX        Mil padre-nossos e mil ave-marias XXI       Prima Justina XXII      Sensações alheias XXIII     Prazo dado XXIV      De mãe e de servo XXV       No Passeio Publico XXVI      As leis são bellas XXVII     Ao portão XXVIII    Na rua XXIX      O imperador XXX       O Santissimo XXXI      As curiosidades de Capitú XXXII     Olhos de ressaca XXXIII    O penteado XXXIV     Sou homem! XXXV      O protonotario apostolico XXXVI     Ideia sem pernas e ideia sem braços XXXVII 

In [29]:
# 6. RANQUEAMENTO SIMPLES (ts_rank) - Frequência da palavra 'saudade'
q6 = """
SELECT titulo, conteudo, ts_rank(ts_conteudo, to_tsquery('portuguese', 'saudade')) as rank
FROM documentos_texto
WHERE ts_conteudo @@ to_tsquery('portuguese', 'saudade')
ORDER BY rank DESC;
"""
rodar_consulta("6. Ranqueamento por Frequência (saudade)", q6)


--- 6. Ranqueamento por Frequência (saudade) ---
('Memórias Póstumas de Brás Cubas - Parágrafo 175', "E foi assim que desembarquei em Lisboa e segui para Coimbra. A Universidade esperava-me com as suas materias arduas, e não sei se profundas; estudei-as muito mediocremente, e nem por isso perdi o gráu de bacharel; deram-m'o com a solemnidade do estylo, após os annos da lei; uma bella festa que me encheu de orgulho e de saudades,--principalmente de saudades. Tinha eu conquistado em Coimbra uma grande nomeada de folião; era um academico estroina, superficial, tumultuario e petulante, dado ás aventuras, fazendo romantismo pratico e liberalismo theorico, vivendo na pura fé dos olhos pretos e das constituições escriptas. No dia em que a Universidade me attestou, em pergaminho, uma sciencia que eu estava longe de trazer arraigada no cerebro, confesso que me achei de de algum modo logrado, ainda que orgulhoso. Explico-me: o diploma era uma carta de alforria; se me dava a liberdade, dava-me a

In [ ]:
# 7. RANQUEAMENTO POR PROXIMIDADE (ts_rank_cd) - 'vício' E 'virtude' próximos
q7 = """
SELECT titulo, conteudo, ts_rank_cd(ts_conteudo, to_tsquery('portuguese', 'vício & virtude')) as rank_cd
FROM documentos_texto
WHERE ts_conteudo @@ to_tsquery('portuguese', 'vício & virtude')
ORDER BY rank_cd DESC;
"""
rodar_consulta("7. Ranqueamento por Proximidade (vício E virtude)", q7)

In [ ]:
# 8. DESTAQUE (ts_headline) - Destacar a palavra 'vida' com marcação HTML
q8 = """
SELECT titulo, ts_headline('portuguese', conteudo, to_tsquery('portuguese', 'vida'))
FROM documentos_texto
WHERE ts_conteudo @@ to_tsquery('portuguese', 'vida');
"""
rodar_consulta("8. Destaque de Termos (Palavra: vida)", q8)